In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix
)

In [ ]:
df = pd.read_csv("../Data/Raw/triage_cleaned_working.csv")
approved_registry = pd.read_csv("../Data/Raw/approved_slice_registry_FINAL.csv")

print(df.shape)
print(approved_registry.shape)

approved_registry.head()

Keep only final accepted slices

In [ ]:
accepted_slices = approved_registry.loc[
    approved_registry["final_accept"].astype(str).str.lower().isin(["yes", "true", "1"]),
    "slice_name"
].tolist()

print("Number of accepted slices:", len(accepted_slices))
print(accepted_slices)

Recreate the exact train / val / test split

In [ ]:
target_col = "y"

model_df = df.copy()

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df[target_col],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[target_col],
    random_state=42
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
all_slice_defs = {
    # --- Clinical / operational slices ---
    "chest_pain": lambda d: d["chest_pain"] == 1,
    "abdominal_pain": lambda d: d["abdominal_pain"] == 1,
    "back_pain": lambda d: d["back_pain"] == 1,
    "headache": lambda d: d["headache"] == 1,
    "cough": lambda d: d["cough"] == 1,
    "unclassified": lambda d: d["unclassified"] == 1,
    "ckd": lambda d: d["ckd"] == 1,
    "dm": lambda d: d["dm"] == 1,
    "htn": lambda d: d["htn"] == 1,
    "chd": lambda d: d["chd"] == 1,
    "pneumonia": lambda d: d["pneumonia"] == 1,
    "arrival_mode_ambulance": lambda d: d["arrival_mode"] == "ambulance",
    "arrival_mode_wheelchair": lambda d: d["arrival_mode"] == "Wheelchair",
    "arrival_mode_unknown": lambda d: d["arrival_mode"] == "Unknown",
    "vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
    "spo2_missing": lambda d: d["spo2_missing"] == 1,
    "hr_missing": lambda d: d["hr_missing"] == 1,
    "tachycardia": lambda d: d["tachycardia_flag"] == 1,
    "hypotension": lambda d: d["hypotension_flag"] == 1,
    "tachypnea": lambda d: d["tachypnea_flag"] == 1,
    "hypoxia": lambda d: d["hypoxia_flag"] == 1,
    "fever_temp": lambda d: d["fever_temp_flag"] == 1,
    "chest_pain_and_ambulance": lambda d: (d["chest_pain"] == 1) & (d["arrival_mode"] == "ambulance"),
    "abdominal_pain_and_hr_missing": lambda d: (d["abdominal_pain"] == 1) & (d["hr_missing"] == 1),
    "ckd_and_spo2_missing": lambda d: (d["ckd"] == 1) & (d["spo2_missing"] == 1),
    "pneumonia_and_tachypnea": lambda d: (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1),
    "unclassified_and_vitals_missing_ge_3": lambda d: (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 3),

    # --- Error-guided slices ---
    "err_vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
    "err_spo2_missing_and_unclassified": lambda d: (d["spo2_missing"] == 1) & (d["unclassified"] == 1),
    "err_hr_missing_and_abdominal_pain": lambda d: (d["hr_missing"] == 1) & (d["abdominal_pain"] == 1),
    "err_pneumonia_and_tachypnea": lambda d: (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1),
    "err_ckd_and_spo2_missing": lambda d: (d["ckd"] == 1) & (d["spo2_missing"] == 1),
    "err_chest_pain_and_tachycardia": lambda d: (d["chest_pain"] == 1) & (d["tachycardia_flag"] == 1),
    "err_unclassified_and_missingness": lambda d: (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 2),
    "err_hypoxia_and_arrival_missing": lambda d: (d["hypoxia_flag"] == 1) & (d["arrival_mode_missing"] == 1),
}

Filter to accepted slices that exist in the registry

In [ ]:
accepted_slices = [s for s in accepted_slices if s in all_slice_defs]

print("Accepted slices found in registry:", len(accepted_slices))
print(accepted_slices)

Add slice membership columns to train/val/test

In [ ]:
for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    for s in accepted_slices:
        split_df[f"SLICE__{s}"] = all_slice_defs[s](split_df).astype(int)

print("Done adding slice membership columns.")

Quick slice coverage audit

In [ ]:
coverage_rows = []

for s in accepted_slices:
    coverage_rows.append({
        "slice_name": s,
        "train_n": train_df[f"SLICE__{s}"].sum(),
        "val_n": val_df[f"SLICE__{s}"].sum(),
        "test_n": test_df[f"SLICE__{s}"].sum(),
        "train_pct": 100 * train_df[f"SLICE__{s}"].mean(),
        "val_pct": 100 * val_df[f"SLICE__{s}"].mean(),
        "test_pct": 100 * test_df[f"SLICE__{s}"].mean(),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values("train_n", ascending=False)
coverage_df

Define the same feature groups

In [ ]:
numeric_cols = [
    "age", "n_admissions",
    "heart_rate_final", "bp_systolic", "bp_diastole",
    "respiratory", "temperature", "spo2",
    "vitals_missing_count"
]

categorical_cols = [
    "arrival_mode", "arrival_month", "arrival_day", "arrival_hour_bin"
]

binary_cols = [
    "abdominal_pain", "chest_pain", "back_pain", "unclassified",
    "ckd", "dm", "headache", "htn", "cough", "chd", "asthma",
    "bronchitis", "dizziness", "glaucoma", "hepatitis", "paralysis",
    "pneumonia", "sicklecell", "arm_pain", "breast_pain", "dental_pain",
    "diarrhea", "ear_pain", "eye_pain", "fever", "pelvic_pain",
    "arrival_mode_missing",
    "hr_missing", "pulse_rate_missing", "bp_systolic_missing",
    "bp_diastole_missing", "respiratory_missing", "temperature_missing", "spo2_missing",
    "heart_rate_disagreement_flag"
]

feature_cols = numeric_cols + categorical_cols + binary_cols

Build the preprocessing pipeline helper

In [ ]:
def make_pipeline():
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    binary_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
            ("bin", binary_transformer, binary_cols),
        ]
    )

    model = LogisticRegression(
        max_iter=3000,
        solver="lbfgs",
        random_state=42
    )

    pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])

    return pipeline

Generate OOF train diagnostics for repair logic

In [ ]:
from sklearn.model_selection import StratifiedKFold

X_train = train_df[feature_cols].copy()
y_train = train_df["y"].copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(train_df))

for fold, (tr_idx, oof_idx) in enumerate(skf.split(X_train, y_train), start=1):
    X_tr = X_train.iloc[tr_idx]
    y_tr = y_train.iloc[tr_idx]
    X_oof = X_train.iloc[oof_idx]

    fold_pipe = make_pipeline()
    fold_pipe.fit(X_tr, y_tr)
    oof_pred[oof_idx] = fold_pipe.predict_proba(X_oof)[:, 1]

    print(f"Fold {fold} complete")

Add OOF diagnostics to train_df

In [ ]:
train_df = train_df.copy().reset_index(drop=True)

train_df["oof_pred_proba"] = oof_pred
train_df["oof_logloss"] = -(
    train_df["y"] * np.log(np.clip(train_df["oof_pred_proba"], 1e-8, 1-1e-8)) +
    (1 - train_df["y"]) * np.log(np.clip(1 - train_df["oof_pred_proba"], 1e-8, 1-1e-8))
)

train_df["oof_pred_label_05"] = (train_df["oof_pred_proba"] >= 0.5).astype(int)

train_df["false_negative_05"] = (
    (train_df["y"] == 1) & (train_df["oof_pred_label_05"] == 0)
).astype(int)

train_df["high_conf_contradiction"] = (
    ((train_df["oof_pred_proba"] >= 0.80) & (train_df["y"] == 0)) |
    ((train_df["oof_pred_proba"] <= 0.20) & (train_df["y"] == 1))
).astype(int)

loss_cutoff = train_df["oof_logloss"].quantile(0.90)
train_df["high_loss_top10"] = (train_df["oof_logloss"] >= loss_cutoff).astype(int)

train_df.head()

Create baseline sample weights (all ones)

In [ ]:
train_df["w_raw"] = 1.0

Create slice burden score

In [ ]:
slice_cols = [f"SLICE__{s}" for s in accepted_slices]

train_df["slice_burden"] = train_df[slice_cols].sum(axis=1)

train_df["slice_burden"].describe()

Repair Arm A: Slice-aware reweighting

In [ ]:
def burden_to_weight(b):
    if b == 0:
        return 1.0
    elif b == 1:
        return 1.25
    elif b == 2:
        return 1.50
    else:
        return 1.75

train_df["w_slice_reweight"] = train_df["slice_burden"].apply(burden_to_weight).astype(float)

train_df["w_slice_reweight"].describe()

Repair Arm B: Suspicious-label downweighting

In [ ]:
train_df["suspicious_in_slice"] = (
    (train_df["slice_burden"] >= 1) &
    (
        (train_df["high_conf_contradiction"] == 1) |
        (train_df["high_loss_top10"] == 1)
    )
).astype(int)

train_df["w_suspicious_downweight"] = 1.0
train_df.loc[train_df["suspicious_in_slice"] == 1, "w_suspicious_downweight"] = 0.70

train_df["suspicious_in_slice"].mean(), train_df["w_suspicious_downweight"].describe()

Repair Arm C: Combined repair

In [ ]:
train_df["w_combined"] = (
    train_df["w_slice_reweight"] *
    train_df["w_suspicious_downweight"]
)

train_df["w_combined"].describe()

weight normalization

In [ ]:
for col in ["w_raw", "w_slice_reweight", "w_suspicious_downweight", "w_combined"]:
    train_df[col] = train_df[col] / train_df[col].mean()

train_df[["w_raw", "w_slice_reweight", "w_suspicious_downweight", "w_combined"]].mean()

Prepare train / test matrices

In [ ]:
X_train = train_df[feature_cols].copy()
y_train = train_df["y"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["y"].copy()

Fit all 4 models

In [ ]:
models = {}
test_preds = {}

weight_cols = {
    "raw_erm": "w_raw",
    "slice_reweight": "w_slice_reweight",
    "suspicious_downweight": "w_suspicious_downweight",
    "combined_repair": "w_combined"
}

for arm_name, w_col in weight_cols.items():
    print(f"Training {arm_name} ...")
    
    pipe = make_pipeline()
    pipe.fit(X_train, y_train, model__sample_weight=train_df[w_col].values)
    
    models[arm_name] = pipe
    test_preds[arm_name] = pipe.predict_proba(X_test)[:, 1]

print("Done.")

Global evaluation helper

In [ ]:
def evaluate_global(y_true, p, threshold=0.5):
    y_pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    
    return {
        "AUROC": roc_auc_score(y_true, p),
        "AUPRC": average_precision_score(y_true, p),
        "Brier": brier_score_loss(y_true, p),
        "Recall@0.5": recall,
        "Precision@0.5": precision
    }

Accepted-slice evaluation helper

In [ ]:
def evaluate_slice(data, mask, p_col, threshold=0.5):
    sub = data.loc[mask].copy()
    
    if len(sub) == 0:
        return None
    
    y = sub["y"].values
    p = sub[p_col].values
    y_pred = (p >= threshold).astype(int)
    
    auroc = np.nan
    if len(np.unique(y)) > 1:
        auroc = roc_auc_score(y, p)
    
    auprc = average_precision_score(y, p)
    
    tn = fp = fn = tp = 0
    if len(np.unique(y)) > 1:
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    else:
        if y[0] == 1:
            tp = (y_pred == 1).sum()
            fn = (y_pred == 0).sum()
        else:
            tn = (y_pred == 0).sum()
            fp = (y_pred == 1).sum()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    
    return {
        "n": len(sub),
        "support_pct": 100 * len(sub) / len(data),
        "event_rate": y.mean(),
        "AUROC": auroc,
        "AUPRC": auprc,
        "Recall@0.5": recall,
        "Precision@0.5": precision
    }

Attach all model predictions to test_df

In [ ]:
test_eval_df = test_df.copy().reset_index(drop=True)

for arm_name, preds in test_preds.items():
    test_eval_df[f"pred__{arm_name}"] = preds

test_eval_df.head()

Rebuild slice membership on test set (after reset_index)

In [ ]:
for s in accepted_slices:
    test_eval_df[f"SLICE__{s}"] = all_slice_defs[s](test_eval_df).astype(int)

Compute global results table

In [ ]:
global_rows = []

for arm_name in weight_cols.keys():
    metrics = evaluate_global(test_eval_df["y"], test_eval_df[f"pred__{arm_name}"])
    metrics["arm"] = arm_name
    global_rows.append(metrics)

global_results = pd.DataFrame(global_rows)[
    ["arm", "AUROC", "AUPRC", "Brier", "Recall@0.5", "Precision@0.5"]
]

global_results.sort_values("AUPRC", ascending=False)

Compute per-slice results for all arms

In [ ]:
slice_rows = []

for arm_name in weight_cols.keys():
    p_col = f"pred__{arm_name}"
    
    for s in accepted_slices:
        mask = test_eval_df[f"SLICE__{s}"] == 1
        m = evaluate_slice(test_eval_df, mask, p_col=p_col)
        
        if m is not None:
            m["arm"] = arm_name
            m["slice_name"] = s
            slice_rows.append(m)

slice_results = pd.DataFrame(slice_rows)

slice_results.head()

Worst-slice summary by arm

In [ ]:
worst_slice_rows = []

for arm_name in weight_cols.keys():
    sub = slice_results[slice_results["arm"] == arm_name].copy()
    
    worst_slice_rows.append({
        "arm": arm_name,
        "num_slices": len(sub),
        "mean_slice_AUPRC": sub["AUPRC"].mean(),
        "worst_slice_AUPRC": sub["AUPRC"].min(),
        "mean_slice_Recall@0.5": sub["Recall@0.5"].mean(),
        "worst_slice_Recall@0.5": sub["Recall@0.5"].min(),
        "avg_to_worst_AUPRC_gap": sub["AUPRC"].mean() - sub["AUPRC"].min(),
        "avg_to_worst_Recall_gap": sub["Recall@0.5"].mean() - sub["Recall@0.5"].min(),
    })

worst_slice_results = pd.DataFrame(worst_slice_rows)

worst_slice_results

Compare slice-by-slice deltas vs raw ERM

In [ ]:
raw_sub = slice_results[slice_results["arm"] == "raw_erm"][["slice_name", "AUPRC", "Recall@0.5"]].copy()
raw_sub = raw_sub.rename(columns={
    "AUPRC": "raw_AUPRC",
    "Recall@0.5": "raw_Recall@0.5"
})

delta_tables = {}

for arm_name in ["slice_reweight", "suspicious_downweight", "combined_repair"]:
    sub = slice_results[slice_results["arm"] == arm_name][["slice_name", "AUPRC", "Recall@0.5"]].copy()
    merged = raw_sub.merge(sub, on="slice_name", how="inner")
    
    merged["delta_AUPRC_vs_raw"] = merged["AUPRC"] - merged["raw_AUPRC"]
    merged["delta_Recall_vs_raw"] = merged["Recall@0.5"] - merged["raw_Recall@0.5"]
    merged["arm"] = arm_name
    
    delta_tables[arm_name] = merged.sort_values("delta_Recall_vs_raw", ascending=False)

delta_tables["combined_repair"]

Threshold stress test helper

In [ ]:
def worst_slice_summary_at_threshold(test_eval_df, accepted_slices, arm_name, threshold):
    p_col = f"pred__{arm_name}"
    rows = []
    
    for s in accepted_slices:
        mask = test_eval_df[f"SLICE__{s}"] == 1
        m = evaluate_slice(test_eval_df, mask, p_col=p_col, threshold=threshold)
        if m is not None:
            rows.append(m)
    
    sub = pd.DataFrame(rows)
    
    return {
        "arm": arm_name,
        "threshold": threshold,
        "mean_slice_AUPRC": sub["AUPRC"].mean(),
        "worst_slice_AUPRC": sub["AUPRC"].min(),
        "mean_slice_Recall": sub["Recall@0.5"].mean(),
        "worst_slice_Recall": sub["Recall@0.5"].min(),
    }

Run threshold stress test

In [ ]:
threshold_rows = []

for arm_name in weight_cols.keys():
    for th in [0.4, 0.5, 0.6]:
        threshold_rows.append(
            worst_slice_summary_at_threshold(test_eval_df, accepted_slices, arm_name, th)
        )

threshold_results = pd.DataFrame(threshold_rows)
threshold_results

Save all major outputs

In [ ]:
coverage_df.to_csv("nb6_slice_coverage.csv", index=False)
global_results.to_csv("nb6_global_results.csv", index=False)
slice_results.to_csv("nb6_slice_results.csv", index=False)
worst_slice_results.to_csv("nb6_worst_slice_results.csv", index=False)

for arm_name, df_delta in delta_tables.items():
    df_delta.to_csv(f"nb6_delta_vs_raw__{arm_name}.csv", index=False)

threshold_results.to_csv("nb6_threshold_stress_test.csv", index=False)

print("All Notebook 6 outputs saved.")

In [ ]:
# ============================================================
# Export locked-test predictions and accepted-slice membership
# for publication-quality figures in Notebook 7
# ============================================================

prediction_cols = [
    "y",
    "pred__raw_erm",
    "pred__slice_reweight",
    "pred__suspicious_downweight",
    "pred__combined_repair"
]

slice_membership_cols = [
    f"SLICE__{slice_name}"
    for slice_name in accepted_slices
]

export_cols = prediction_cols + slice_membership_cols

missing_cols = [
    col for col in export_cols
    if col not in test_eval_df.columns
]

if missing_cols:
    raise KeyError(
        "The following required columns are missing from test_eval_df: "
        + ", ".join(missing_cols)
    )

test_eval_df[export_cols].to_csv(
    "nb6_test_predictions_with_slices.csv",
    index=False
)

print("Saved: nb6_test_predictions_with_slices.csv")
print("Shape:", test_eval_df[export_cols].shape)